# 7.16 舊例子混回去有用嗎？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**A+B訓練兩倍步數比B-only好，能歸因於replay嗎？

replay把一部分舊資料混進來提醒模型。比較時總更新或有效tokens保持相同，否則額外預算可能是原因。

**把直覺寫成數學：**混合loss的分母依所有有效tokens；樣本比例不直接等於梯度權重。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
a = [{"source": "A", "text": "舊任務例"}] * 2
b = [{"source": "B", "text": "新任務例"}] * 2
recipe = a + b
print("固定batch大小4的replay來源", [r["source"] for r in recipe])
print("待量測：B-only和replay各自的A/B分數、相同有效token預算")

**如何讀結果：**真實實驗用不同原始例子，不能靠重複同一行聲稱資料覆蓋增加。

**只改一件事：**只把來源比例從50:50改爲25:75。
